In [1]:
import pdf_creator as pc

In [2]:
df = pc.create_sample_dataframe()

In [3]:
generator = pc.QuizPDFGenerator(df)
generator.generate_pdf()

PDF generated successfully: quiz.pdf


In [6]:
pip install io

55.03s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
ERROR: Could not find a version that satisfies the requirement io (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
ERROR: No matching distribution found for io
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import dictionary manager
from dictionary_manager import ItalianDictionary, create_github_url

# Initialize dictionary
dict_manager = ItalianDictionary("dictionary.json")

# Show dictionary statistics
print("=== Italian Dictionary Statistics ===")
stats = dict_manager.get_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n=== Most Frequent Words (Top 10) ===")
frequent_words = dict_manager.get_most_frequent_words(10)
for i, word in enumerate(frequent_words, 1):
    translations = dict_manager.search_translations(word)
    print(f"{i:2d}. {word:12} -> {', '.join(translations)}")

print("\n=== Example Searches ===")
# Search for translations
test_words = ["be", "have", "know", "hello"]
for word in test_words:
    translations = dict_manager.search_translations(word)
    if translations:
        print(f"'{word}' -> {translations}")
        
        # Show conjugations for verbs
        if dict_manager.search_word(word).get('type') == 'verb':
            for italian_verb in translations:
                conj = dict_manager.get_conjugations(word, italian_verb)
                if conj:
                    print(f"  {italian_verb} conjugations:")
                    for tense, persons in list(conj.items())[:2]:  # Show first 2 tenses
                        print(f"    {tense}: {persons}")
                    break

=== Italian Dictionary Statistics ===
total_words: 20
by_type: {'verb': 7, 'noun': 9, 'phrase': 1, 'adverb': 1, 'adjective': 2}
by_category: {'essential': 7, 'greetings': 2, 'politeness': 2, 'basic_needs': 3, 'places': 1, 'concepts': 1, 'time': 2, 'descriptions': 2}
verbs_with_conjugations: 7

=== Most Frequent Words (Top 10) ===
 1. be           -> essere
 2. have         -> avere
 3. go           -> andare
 4. do           -> fare
 5. say          -> dire, parlare
 6. see          -> vedere
 7. know         -> sapere, conoscere
 8. hello        -> ciao, salve
 9. goodbye      -> arrivederci, ciao
10. thank you    -> grazie, ti ringrazio

=== Example Searches ===
'be' -> ['essere']
  essere conjugations:
    presente: {'io': 'sono', 'tu': 'sei', 'lui/lei': 'è', 'noi': 'siamo', 'voi': 'siete', 'loro': 'sono'}
    passato_prossimo: {'io': 'sono stato/a', 'tu': 'sei stato/a', 'lui/lei': 'è stato/a', 'noi': 'siamo stati/e', 'voi': 'siete stati/e', 'loro': 'sono stati/e'}
'have' -> ['avere

In [6]:
import pandas as pd

df = pd.read_clipboard(sep=",")

df

,English,Italian_Infinitive,Regularity,1st_Sing,2nd_Sing,3rd_Sing,1st_Plur,2nd_Plur,3rd_Plur
0,be,essere,irregular,sarei,saresti,sarebbe,saremmo,sareste,sarebbero
1,have,avere,irregular,avrei,avresti,avrebbe,avremmo,avreste,avrebbero
2,do,fare,irregular,farei,faresti,farebbe,faremmo,fareste,farebbero
3,say,dire,irregular,direi,diresti,direbbe,diremmo,direste,direbbero
4,get,ottenere,irregular,otterrei,otterresti,otterrebbe,otterremmo,otterreste,otterrebbero
...,...,...,...,...,...,...,...,...,...
96,pick,scegliere,irregular,sceglierei,sceglieresti,sceglierebbe,sceglieremmo,scegliereste,sceglierebbero
97,become,diventare,regular,diventerei,diventeresti,diventerebbe,diventeremmo,diventereste,diventerebbero
98,carry,portare,regular,porterei,porteresti,porterebbe,porteremmo,portereste,porterebbero
99,drive,guidare,regular,guiderei,guideresti,guiderebbe,guideremmo,guidereste,guiderebbero


In [7]:
for i in range(len(df)):
    row = df.loc[i]

row

English                     receive
Italian_Infinitive         ricevere
Regularity                  regular
1st_Sing                  riceverei
2nd_Sing                riceveresti
3rd_Sing                riceverebbe
1st_Plur                riceveremmo
2nd_Plur                ricevereste
3rd_Plur              riceverebbero
Name: 100, dtype: object

In [8]:
df.to_csv('presente_condizionale.csv', index = False)

In [ ]:
# Functions to add new words to dictionary

def add_simple_word(english: str, italian_list: list, word_type: str = "noun", category: str = "user_added", **kwargs):
    """Add a simple word (non-verb) to dictionary"""
    dict_manager.add_word(english, word_type, italian_list, category=category, **kwargs)
    dict_manager.save_dictionary()
    print(f"Added: {english} -> {italian_list}")

def create_conjugation_template():
    """Create a template for Italian verb conjugations"""
    return {
        "io": "",
        "tu": "",
        "lui/lei": "",
        "noi": "",
        "voi": "",
        "loro": ""
    }

def add_verb_with_conjugations(english: str, italian_verb: str, presente_conj: dict, **other_tenses):
    """
    Add verb with conjugations to dictionary
    All conjugations must include: io, tu, lui/lei, noi, voi, loro
    """
    # Validate presente conjugation format
    required_pronouns = {"io", "tu", "lui/lei", "noi", "voi", "loro"}
    if not all(pronoun in presente_conj for pronoun in required_pronouns):
        missing = required_pronouns - set(presente_conj.keys())
        raise ValueError(f"Missing required pronouns in presente: {missing}")
    
    conjugations = {
        italian_verb: {
            "presente": presente_conj
        }
    }
    
    # Add other tenses if provided (also validate them)
    for tense, conj in other_tenses.items():
        if not all(pronoun in conj for pronoun in required_pronouns):
            missing = required_pronouns - set(conj.keys())
            raise ValueError(f"Missing required pronouns in {tense}: {missing}")
        conjugations[italian_verb][tense] = conj
    
    dict_manager.add_word(english, "verb", [italian_verb], conjugations=conjugations, category="user_added")
    dict_manager.save_dictionary()
    print(f"Added verb: {english} -> {italian_verb} with conjugations")

def search_and_display(word: str):
    """Search for word and display all information"""
    print(f"\n=== Search Results for '{word}' ===")
    
    # Search English -> Italian
    translations = dict_manager.search_translations(word)
    if translations:
        word_data = dict_manager.search_word(word)
        print(f"English: {word}")
        print(f"Type: {word_data.get('type', 'unknown')}")
        print(f"Translations: {translations}")
        print(f"Category: {word_data.get('category', 'unknown')}")
        
        if 'notes' in word_data:
            print(f"Notes: {word_data['notes']}")
        
        # Show conjugations for verbs
        if word_data.get('type') == 'verb' and 'conjugations' in word_data:
            print("Conjugations:")
            for italian_verb, tenses in word_data['conjugations'].items():
                print(f"  {italian_verb}:")
                for tense, persons in tenses.items():
                    print(f"    {tense}: {persons}")
    
    # Search Italian -> English  
    english_matches = dict_manager.search_by_italian(word)
    if english_matches:
        print(f"Italian '{word}' found in: {english_matches}")
    
    if not translations and not english_matches:
        print(f"'{word}' not found in dictionary")

# Example usage:
print("=== Dictionary Management Functions Loaded ===")
print("Available functions:")
print("- add_simple_word(english, italian_list, word_type, category)")
print("- create_conjugation_template() -> returns empty conjugation dict")
print("- add_verb_with_conjugations(english, italian_verb, presente_conj, **other_tenses)")  
print("- search_and_display(word)")
print("- dict_manager.export_to_csv(filename)")
print("- dict_manager.get_stats()")

print("\n=== Required Conjugation Format ===")
print("All verbs must include these pronouns:")
print("io, tu, lui/lei, noi, voi, loro")

print("\nExample verb addition:")
print("""
# Create conjugation template
conj = create_conjugation_template()
conj["io"] = "mangio"
conj["tu"] = "mangi"  
conj["lui/lei"] = "mangia"
conj["noi"] = "mangiamo"
conj["voi"] = "mangiate"
conj["loro"] = "mangiano"

add_verb_with_conjugations("eat", "mangiare", conj)
""")

In [ ]:
# GitHub Integration
# Once you push to GitHub, you can access the dictionary via raw URL

github_username = "Norris36"
github_repo = "italiano" 
github_url = create_github_url(github_username, github_repo, "dictionary.json", "dev")

print(f"GitHub raw URL for dictionary: {github_url}")
print("\nTo use dictionary from GitHub:")
print(f"dict_github = ItalianDictionary('{github_url}')")
print("\nThis allows accessing your dictionary from anywhere via the web!")

# Create a backup and export functions
def backup_and_export():
    """Create backups and exports of dictionary"""
    dict_manager.save_dictionary("dictionary_backup.json")
    dict_manager.export_to_csv("italian_words.csv") 
    dict_manager.export_conjugations_csv("italian_conjugations.csv")
    print("✓ Created dictionary_backup.json")
    print("✓ Exported to italian_words.csv")  
    print("✓ Exported conjugations to italian_conjugations.csv")

print("\nRun backup_and_export() to create backups and CSV exports")

In [ ]:
# Version Management and Project Info
from version import VersionManager, get_version

# Display version information
version_manager = VersionManager()
version_info = version_manager.get_version_info()

print("🇮🇹 ITALIAN DICTIONARY PROJECT 🇮🇹")
print("=" * 50)
print(f"📊 Version: {version_info['version']}")
print(f"📅 Updated: {version_info['date']}")
print(f"📝 Description: {version_info['description']}")

# Dictionary disclaimer
print("\n⚠️  DISCLAIMER:")
print("   Translations are not 100% accurate and are being")
print("   updated iteratively as learning progresses.")
print("   Please verify important translations with authoritative sources.")

# Project goals
print(f"\n🎯 PROJECT GOALS:")
print(f"   Current: {dict_manager.get_stats()['total_words']} words")
print(f"   Target: 1000 words (following Pareto principle)")
print(f"   Focus: High-frequency words for 80% communication coverage")

print("\n" + "=" * 50)

In [7]:
import dictionary_manager as dm

dm.main()

Dictionary Statistics:
  total_words: 20
  by_type: {'verb': 7, 'noun': 9, 'phrase': 1, 'adverb': 1, 'adjective': 2}
  by_category: {'essential': 7, 'greetings': 2, 'politeness': 2, 'basic_needs': 3, 'places': 1, 'concepts': 1, 'time': 2, 'descriptions': 2}
  verbs_with_conjugations: 7

Search Examples:
'be' -> ['essere']
Conjugations for 'essere':
  presente: {'io': 'sono', 'tu': 'sei', 'lui/lei': 'è', 'noi': 'siamo', 'voi': 'siete', 'loro': 'sono'}
  passato_prossimo: {'io': 'sono stato/a', 'tu': 'sei stato/a', 'lui/lei': 'è stato/a', 'noi': 'siamo stati/e', 'voi': 'siete stati/e', 'loro': 'sono stati/e'}
  imperfetto: {'io': 'ero', 'tu': 'eri', 'lui/lei': 'era', 'noi': 'eravamo', 'voi': 'eravate', 'loro': 'erano'}
Words that translate to 'ciao': ['hello', 'goodbye']


In [8]:
dm.create_github_url('norris36','italiano', 'dictionary.json', 'dev')

'https://raw.githubusercontent.com/norris36/italiano/dev/dictionary.json'